In [ ]:
# Runtime setup for the Chapter 9 kernel listings.
import numpy as np
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=96, noise=0.20, random_state=7)

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=7
)

def linear_gram(A, B):
    return A @ B.T

K_train = linear_gram(X_train, X_train)
K_test = linear_gram(X_test, X_train)

classifier = SVC(kernel="precomputed", C=1.0)
classifier.fit(K_train, y_train)

predictions = classifier.predict(K_test)
score = balanced_accuracy_score(y_test, predictions)
print(score)

In [ ]:
# Keep raw split names for the baseline-comparison listing.
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

In [ ]:
# Minimal analytic stand-ins for the framework-neutral quantum-kernel sketch.
selected_C = 1.0

def feature_map(x):
    x = np.asarray(x, dtype=float)
    state = np.array([
        np.cos(x[0]),
        np.sin(x[0]),
        np.cos(x[1]),
        np.sin(x[1]),
    ], dtype=float)
    return state / np.linalg.norm(state)

def overlap_circuit(feature_map, x_left, x_right):
    return (
        feature_map,
        np.asarray(x_left, dtype=float),
        np.asarray(x_right, dtype=float),
    )

class AnalyticOverlapBackend:
    def estimate_all_zero_probability(self, circuit):
        mapper, x_left, x_right = circuit
        left_state = mapper(x_left)
        right_state = mapper(x_right)
        fidelity = np.dot(left_state, right_state) ** 2
        return float(np.clip(fidelity, 0.0, 1.0))

backend = AnalyticOverlapBackend()

In [ ]:
def evaluate_quantum_kernel(A, B, feature_map, backend):
    K = np.empty((len(A), len(B)))
    for i, x_left in enumerate(A):
        for j, x_right in enumerate(B):
            circuit = overlap_circuit(feature_map, x_left, x_right)
            K[i, j] = backend.estimate_all_zero_probability(circuit)
    return K

K_train = evaluate_quantum_kernel(X_train, X_train, feature_map, backend)
K_test = evaluate_quantum_kernel(X_test, X_train, feature_map, backend)

classifier = SVC(kernel="precomputed", C=selected_C)
classifier.fit(K_train, y_train)
predictions = classifier.predict(K_test)

In [ ]:
# Preserve explicit names for the baseline-comparison listing.
K_train_quantum = K_train
K_test_quantum = K_test

In [ ]:
def kernel_diagnostics(K, y):
    K = 0.5 * (K + K.T)
    y_pm = np.where(y > 0, 1.0, -1.0)

    same = y_pm[:, None] == y_pm[None, :]
    off_diagonal = ~np.eye(len(y), dtype=bool)

    same_mean = K[same & off_diagonal].mean()
    diff_mean = K[(~same) & off_diagonal].mean()

    H = np.eye(len(y)) - np.ones_like(K) / len(y)
    K_centered = H @ K @ H
    Y_centered = H @ np.outer(y_pm, y_pm) @ H
    alignment = np.sum(K_centered * Y_centered) / (
        np.linalg.norm(K_centered) * np.linalg.norm(Y_centered)
    )

    eigenvalues = np.linalg.eigvalsh(K)
    return {
        "same_mean": same_mean,
        "diff_mean": diff_mean,
        "similarity_gap": same_mean - diff_mean,
        "alignment": alignment,
        "min_eigenvalue": eigenvalues.min(),
        "max_eigenvalue": eigenvalues.max(),
    }

In [ ]:
kernel_diagnostics(K_train_quantum, y_train)

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

classical_models = {
    "linear": make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0)),
    "rbf": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale")),
    "poly2": make_pipeline(StandardScaler(), SVC(kernel="poly", degree=2, C=1.0)),
}

for name, model in classical_models.items():
    model.fit(X_train_raw, y_train)
    y_pred = model.predict(X_test_raw)
    print(name, balanced_accuracy_score(y_test, y_pred))

quantum_svc = SVC(kernel="precomputed", C=1.0)
quantum_svc.fit(K_train_quantum, y_train)
quantum_pred = quantum_svc.predict(K_test_quantum)
print("quantum", balanced_accuracy_score(y_test, quantum_pred))